In [1]:
using ArviZ
using Bijectors
using CairoMakie
using Colors
using DataFrames
using DimensionalData
using Distributions
using FITSIO
using HDF5
using LinearAlgebra
using Mooncake
using PairPlots
using PulsarLightcurveExtraction
using StatsFuns
using Turing


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
BLAS.set_num_threads(1)
BLAS.get_num_threads()

1

In [3]:
function husl_wheel(n)
    return [LCHuv(65, 90, h) for h in range(0, stop=360, length=n+1)][1:end-1]
end

function traceplot(chain; var_names=nothing)
    if var_names !== nothing
        p = chain.posterior[var_names]
    else
        p = chain.posterior
    end

    vars = keys(p)

    f = Figure(size=(800, 200*length(vars)))
    for (i, var) in enumerate(vars)
        zs = p[var]
        c = dims(zs, :chain)
        ds = otherdims(zs, (:chain, :draw))
        
        all_ds = (c, ds...)

        zs_merge = mergedims(zs, all_ds => :merged_chain)
        n = size(zs_merge, :merged_chain)

        adens = Axis(f[i,1], xlabel=String(var), palette=(color = husl_wheel(n), patchcolor = husl_wheel(n),))
        atrace = Axis(f[i,2], xlabel="Iteration", ylabel=String(var), palette=(color = husl_wheel(n), patchcolor = husl_wheel(n),))

        for c in dims(zs_merge, :merged_chain)
            density!(adens, vec(zs_merge[merged_chain=At(c)]); label=nothing)
            lines!(atrace, vec(zs_merge[merged_chain=At(c)]); label=nothing)
        end
    end

    return f
end

traceplot (generic function with 1 method)

In [4]:
event_time, event_phase, event_pi, segment_start, segment_stop = FITS(joinpath(@__DIR__, "..", "data", "J0740_merged_phase_0.25-3keV.fits.gz"), "r") do f
    event_time = read(f[2], "TIME")
    event_phase = read(f[2], "PULSE_PHASE")
    event_pi = read(f[2], "PI")

    segment_start = read(f[4], "START")
    segment_stop = read(f[4], "STOP")

    (event_time, event_phase, event_pi, segment_start, segment_stop)
end

segment_Ts = segment_stop .- segment_start;

In [5]:
arf_e_high, arf_e_low, arf_e, arf_response, arf_start, arf_stop = h5open(joinpath(@__DIR__, "..", "data", "J0740_merged_arf.h5"), "r") do f
    map(x -> read(f, x), ("ENERG_HI", "ENERG_LO", "ENERGY", "SPECRESP", "START", "STOP"))
end

(Float32[0.105, 0.11, 0.115, 0.12, 0.125, 0.13, 0.135, 0.14, 0.145, 0.15  …  19.55, 19.6, 19.65, 19.7, 19.75, 19.8, 19.85, 19.9, 19.95, 20.0], Float32[0.1, 0.105, 0.11, 0.115, 0.12, 0.125, 0.13, 0.135, 0.14, 0.145  …  19.5, 19.55, 19.6, 19.65, 19.7, 19.75, 19.8, 19.85, 19.9, 19.95], Float32[0.1025, 0.1075, 0.1125, 0.1175, 0.1225, 0.1275, 0.1325, 0.1375, 0.1425, 0.14750001  …  19.525, 19.575, 19.625, 19.675, 19.725, 19.775, 19.825, 19.875, 19.925, 19.975], Float32[117.835526 117.86762 … 112.862976 112.94827; 122.74665 122.78008 … 117.56688 117.65574; … ; 0.138463 0.13850175 … 0.13266586 0.13276659; 0.13587011 0.13590814 … 0.13018116 0.13028003], [1.49023005e8, 1.49039675e8, 1.49289705e8, 1.49328611e8, 1.49411925e8, 1.49600984e8, 1.50100588e8, 1.50428318e8, 1.50511528e8, 1.50600409e8  …  3.40584146e8, 3.40779302e8, 3.41370902e8, 3.43637612e8, 3.43721249e8, 3.5166214e8, 3.52201587e8, 3.57661926e8, 3.58186873e8, 3.58214905e8], [1.49036122e8, 1.49102802e8, 1.49297562e8, 1.4938092e8, 1.49447

In [6]:
n_spec = 50
n_segments = 10 # nothing
n_fourier = 10

cm, sm = PulsarLightcurveExtraction.cos_sin_matrices(event_phase, n_fourier)
m = cat(cm, sm, dims=2)

event_segment_indices = PulsarLightcurveExtraction.segment_indices(event_time, segment_start, segment_stop);
event_areas = PulsarLightcurveExtraction.event_areas(event_time, event_pi, arf_start, arf_stop, arf_e_low, arf_e_high, arf_response);
event_spectral_indices, spec_bins_pi = PulsarLightcurveExtraction.spectral_indices(event_pi, n_spec)

spec_bin_centers = 0.5 .* (spec_bins_pi[1:end-1] .+ spec_bins_pi[2:end]) .* PulsarLightcurveExtraction.PI_TO_KEV

# Scale design matrix by event areas, so that the fg_coeffs are in units of counts per square cm per second.
m = m .* reshape(event_areas, (:, 1))

if n_segments != nothing
    event_sel = event_segment_indices .<= n_segments

    event_time = event_time[event_sel]
    event_segment_indices = event_segment_indices[event_sel]
    event_areas = event_areas[event_sel]
    event_spectral_indices = event_spectral_indices[event_sel]
    m = m[event_sel, :]
end

segment_start = segment_start[1:n_segments]
segment_stop = segment_stop[1:n_segments]
segment_Ts = segment_Ts[1:n_segments];

In [7]:
est_log_bg, est_log_bg_uncert = PulsarLightcurveExtraction.estimate_log_bg(event_time, segment_start, segment_stop)

mean_est_log_bg = mean(est_log_bg)
std_est_log_bg = std(est_log_bg)

println("Estimated log background rate across segments: $(mean_est_log_bg) ± $(std_est_log_bg)")

Estimated log background rate across segments: -0.2260679076309476 ± 0.8110934844634247


In [8]:
est_bg_spec, est_bg_spec_uncert = PulsarLightcurveExtraction.estimate_bg_spec(event_spectral_indices);

In [9]:
fg_scale = 5e-6 # Empirically determined fg rate estimate, based on not constraining the posterior too much.

model = PulsarLightcurveExtraction.spec_fourier_model(m, event_segment_indices, event_spectral_indices, segment_Ts, est_log_bg, est_log_bg_uncert, est_bg_spec, est_bg_spec_uncert, fg_scale)

DynamicPPL.Model{typeof(spec_fourier_model), (:design_matrix, :event_segment_indices, :event_spectral_indices, :segment_Ts, :est_log_bg, :est_log_bg_uncert, :est_bg_spec, :est_bg_spec_uncert, :fg_scale), (), (), Tuple{Matrix{Float64}, Vector{Int64}, Vector{Int64}, Vector{Float64}, Vector{Float64}, Vector{Float64}, Vector{Float64}, Vector{Float64}, Float64}, Tuple{}, DynamicPPL.DefaultContext, false}(PulsarLightcurveExtraction.spec_fourier_model, (design_matrix = [843.9821722044169 397.95409160735585 … -869.4006171392086 -441.52435070548916; -852.3161177257155 400.2015208102533 … -873.658580668189 436.5864463603417; … ; 946.0289699478045 725.5658378693607 … 380.4320952569019 714.0910222947847; 768.2425418559545 -275.0039947601874 … 1200.98058868898 531.4942060799237], event_segment_indices = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  10, 10, 10, 10, 10, 10, 10, 10, 10, 10], event_spectral_indices = [6, 4, 3, 11, 6, 34, 7, 16, 6, 6  …  46, 27, 12, 41, 23, 23, 4, 4, 6, 7], segment_Ts = [1997.0, 10

In [ ]:
nmcmc = 100
chains = sample(model, NUTS(nmcmc, 0.65; adtype=AutoMooncake()), nmcmc; initial_params = InitFromParams((fg_coeffs = zeros(size(m,2)), )))

In [ ]:
describe(chains)

In [ ]:
trace = from_mcmcchains(chains; 
                        dims=Dict(:fg_coeffs => (:fourier,), :log_dbg_segment => (:segment,), :log_bg_segment => (:segment,), :bg_segment => (:segment,), :bg_spec => (:energy,), :fg_spec => (:energy,)),
                        coords=Dict(:fourier => 1:size(m,2), :segment => 1:n_segments, :energy => spec_bin_centers))

In [ ]:
traceplot(trace)

In [ ]:
n_lc = 100
phases = range(0, stop=1, length=n_lc)
cm_lc, sm_lc = PulsarLightcurveExtraction.cos_sin_matrices(phases, n_fourier)
m_lc = DimArray(cat(cm_lc, sm_lc, dims=2), (phases=phases, fourier=1:2*n_fourier));

In [ ]:
lcs = @d m_lc .* trace.posterior.fg_coeffs
lcs = dropdims(sum(lcs, dims=:fourier), dims=:fourier)

In [ ]:
lc_m = dropdims(mean(lcs, dims=(:chain, :draw)), dims=(:chain, :draw))
lc_s = dropdims(std(lcs, dims=(:chain, :draw)), dims=(:chain, :draw))

In [ ]:
f = Figure()
a = Axis(f[1,1], xlabel="Phase", ylabel="Counts per second per square cm")

scatter!(a, phases, vec(lc_m))
errorbars!(a, phases, vec(lc_m), vec(lc_s))

f

In [ ]:
fg_spec_m = dropdims(mean(trace.posterior.fg_spec, dims=(:chain, :draw)), dims=(:chain, :draw))
fg_spec_s = dropdims(std(trace.posterior.fg_spec, dims=(:chain, :draw)), dims=(:chain, :draw))

bg_spec_m = dropdims(mean(trace.posterior.bg_spec, dims=(:chain, :draw)), dims=(:chain, :draw))
bg_spec_s = dropdims(std(trace.posterior.bg_spec, dims=(:chain, :draw)), dims=(:chain, :draw));

In [ ]:
f = Figure()
a = Axis(f[1,1], xlabel="Energy (keV)", ylabel="Spectral Probability")

eg = vec(spec_bin_centers)
 
c = Makie.wong_colors()[1]
errorbars!(a, spec_bin_centers, vec(fg_spec_m), vec(fg_spec_s); label="Foreground", color=c)
scatter!(a, spec_bin_centers, vec(fg_spec_m); color=c)

c = Makie.wong_colors()[2]
errorbars!(a, spec_bin_centers, vec(bg_spec_m), vec(bg_spec_s); label="Background", color=c)
scatter!(a, spec_bin_centers, vec(bg_spec_m); color=c)

axislegend(a)

f